# Get Chart Ground Truth

In [1]:
import numpy as np
from tqdm import tqdm
import glob
import re
import json
from typing import Dict, List, Optional, Tuple, Union
import os
import inspect
from copy import deepcopy

import pandas as pd

In [2]:
def get_gt(file_path:str, word_num:int, 
           rename = {'Image.no':'image_no','print size':'print_size'},
           L_R:bool = False, human=False) -> pd.DataFrame:
    # Specify the path to your Excel file
    
    # Read the Excel file into a pandas DataFrame
    xls = pd.ExcelFile(file_path)
    # 获取所有sheet的名称
    sheet_names = xls.sheet_names

    # 逐个读取sheet，并存入列表
    dfs = [pd.read_excel(xls, sheet_name=sheet) for sheet in sheet_names]

    # 合并所有的DataFrame
    gt = pd.concat(dfs, ignore_index=True)
    
    # df = pd.read_excel(file_path, sheet_name=None, header=0)
    
    cols = ['SubID','print_size','image_no','a','b'] + ['L{}'.format(num) for num in range(1,word_num+1)]
    # if human:
    #     cols = ['SubID'] + cols
    
    if L_R:
        cols += ['R{}'.format(num) for num in range(1,word_num+1)]

    # gt = pd.concat(df,axis=0,ignore_index=True)
    if rename is not None:
        gt.rename(columns={'Image.no':'image_no','print size':'print_size'}, inplace=True)
    gt['print_size'] = np.round(gt['print_size'],decimals=1)
    gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
    
    cols = ['row'] + cols
    gt = gt.loc[:,cols].set_index(['print_size','image_no'])

    
    if human:
        return gt


    # 找到重复的索引
    duplicate_index = gt.index.duplicated()

    # 过滤出不重复的行
    df_no_duplicates = gt[~duplicate_index]
    return df_no_duplicates

In [73]:
import pandas as pd
import json

def dataframe_to_coco(df):
    """
    Convert a DataFrame to COCO captioning format.
    
    Assumptions:
      - The image path is constructed by concatenating the values from the "a" and "row" columns.
      - The caption is built by concatenating the text from columns L1 to L15 (ignoring NaNs).
    """
    coco = {
        "images": [],
        "annotations": [],
        "info": {
            "description": "MNRead Dataset",
            "version": "1.0"
        },
        "licenses": []
    }
    
    image_id = 0
    annotation_id = 0
    word_num = max([int(i[1:]) for i in df.columns if 'L' in i])

    # Iterate through each row in the dataframe
    for _, row in df.iterrows():
        # Construct the image file name. Here we assume column 'a' is the image number
        # and 'row' is another identifier. Adjust formatting as needed.
        file_name = f"{row['image_no']}_2260_{int(row['row'])}.jpg"
        
        # Build caption by concatenating L1 to L15 (ignoring NaN values)
        caption_parts = []
        for i in range(1, word_num+1):
            col_name = f"L{i}"
            token = row.get(col_name)
            if pd.notnull(token):
                caption_parts.append(str(token))
        caption = " ".join(caption_parts)
        
        # Add image entry (you may add width, height if known)
        coco["images"].append({
            "id": image_id if 'index' not in row else row['index'],
            "file_name": file_name,
            "Filter_no": row['Filter_no']
        })
        
        # Add annotation entry for the caption
        coco["annotations"].append({
            "id": annotation_id if 'index' not in row else row['index'],
            "image_id": image_id if 'index' not in row else row['index'],
            "caption": caption
        })
        
        image_id += 1
        annotation_id += 1
        
    return coco



In [5]:
filter_df = pd.read_csv('../../data/human/SelectedFilter.csv')

xlsx_file_path = '../../data/human/MNREADChartScoreSheet.xlsx'
WORD_NUM = 15
 
df_no_duplicates = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM)
df_no_duplicates.drop(columns=['SubID'], inplace=True)
df_no_duplicates['a'] = df_no_duplicates['a'].round(3)
df_no_duplicates['b'] = df_no_duplicates['b'].round(3)
df_no_duplicates['a'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['b'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['a'].replace(0.287, 0.288, inplace=True)
df_no_duplicates['b'].replace(0.287, 0.288, inplace=True)
df_no_duplicates.reset_index(inplace=True)
df_no_duplicates = pd.merge(left=df_no_duplicates,right=filter_df[['a','b','Filter_no']],how='inner',on=['a','b'])

/tmp/ipykernel_3157193/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/tmp/ipykernel_3157193/3183782216.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' o

In [78]:
coco_dict = dataframe_to_coco(df_no_duplicates)
    
# Save the COCO JSON to a file
with open("../../data/mnread/anno.json", "w") as f:
    json.dump(coco_dict, f, indent=4)
    

In [ ]:
# xlsx_file_path = '../../data/human/LetterChartScoreSheet.xlsx'
# WORD_NUM = 5
 
# df_no_duplicates = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM)
# df_no_duplicates.drop(columns=['SubID'], inplace=True)
# df_no_duplicates['a'] = df_no_duplicates['a'].round(3)
# df_no_duplicates['b'] = df_no_duplicates['b'].round(3)
# df_no_duplicates['a'].replace(0.156, 0.157, inplace=True)
# df_no_duplicates['b'].replace(0.156, 0.157, inplace=True)
# df_no_duplicates['a'].replace(0.287, 0.288, inplace=True)
# df_no_duplicates['b'].replace(0.287, 0.288, inplace=True)
# df_no_duplicates.reset_index(inplace=True)
# df_no_duplicates = pd.merge(left=df_no_duplicates,right=filter_df[['a','b','Filter_no']],how='inner',on=['a','b'])
# coco_dict = dataframe_to_coco(df_no_duplicates)
    
# # Save the COCO JSON to a file
# with open("../../data/etdrs/anno.json", "w") as f:
#     json.dump(coco_dict, f, indent=4)

# Update ETDRS GT
- the forth line of every chart is missing

In [80]:
def replace_rows(group):
    """
    For each group (chart), if the maximum 'row' equals 21,
    then starting from the row where 'row' is >= 4 and for each row (except the last),
    replace its L1-L5 values with the values from the next row.
    """
    # Sort by the 'row' column (if not already sorted)
    group = group.sort_values(by='row').reset_index(drop=True)
    # Check if last row (i.e. max row number) is 21.
    if group['row'].max() == 21:
        # Iterate over rows from the beginning to the second last row.
        for i in range(len(group) - 1):
            # Check if the current row's 'row' value is >= 4.
            if group.loc[i, 'row'] >= 4:
                # Replace L1 to L5 with the values from the next row.
                group.loc[i, ['L1', 'L2', 'L3', 'L4', 'L5']] = group.loc[i+1, ['L1', 'L2', 'L3', 'L4', 'L5']]
    if group.loc[0, ['L1', 'L2', 'L3', 'L4', 'L5']].isna().any():
        for i in range(len(group) - 2):
            # If any of L1-L5 is NaN in the current row
            if group.loc[i, ['L1', 'L2', 'L3', 'L4', 'L5']].isna().any():
                # Replace this row's L1-L5 with the next row's values
                group.loc[i, ['L1', 'L2', 'L3', 'L4', 'L5']] = group.loc[i+2, ['L1', 'L2', 'L3', 'L4', 'L5']]
                group.loc[i+2, ['L1', 'L2', 'L3', 'L4', 'L5']] = np.nan
    return group

In [82]:
xlsx_file_path = '../../data/human/LetterChartScoreSheet.xlsx'
WORD_NUM = 5
 
df_no_duplicates = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM)
df_no_duplicates.drop(columns=['SubID'], inplace=True)
df_no_duplicates['a'] = df_no_duplicates['a'].round(3)
df_no_duplicates['b'] = df_no_duplicates['b'].round(3)
df_no_duplicates['a'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['b'].replace(0.156, 0.157, inplace=True)
df_no_duplicates['a'].replace(0.287, 0.288, inplace=True)
df_no_duplicates['b'].replace(0.287, 0.288, inplace=True)
df_no_duplicates.reset_index(inplace=True)
df_no_duplicates = pd.merge(left=df_no_duplicates,right=filter_df[['a','b','Filter_no']],how='inner',on=['a','b'])
# Group by 'image_no' (each chart) and apply the replacement function.
df_no_duplicates = df_no_duplicates.reset_index()
df_no_duplicates = df_no_duplicates.groupby('image_no').apply(replace_rows).reset_index(drop=True)
df_no_duplicates = df_no_duplicates[df_no_duplicates['row'] != 21]
df_no_duplicates = df_no_duplicates.set_index('index').sort_index()
coco_dict = dataframe_to_coco(df_no_duplicates.dropna().reset_index())
    
# Save the COCO JSON to a file
with open("../../data/etdrs/anno.json", "w") as f:
    json.dump(coco_dict, f, indent=4)


/tmp/ipykernel_3157193/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/tmp/ipykernel_3157193/3987458473.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or

# Human Acuity on differnt filters

In [ ]:
from collections import Counter
def count_matches_per_row(row):
    left = ['L'+str(i+1) for i in range(WORD_NUM)]
    left_side = row[left]
    right = ['R'+str(i+1) for i in range(WORD_NUM)]
    right_side = row[right]
    
    counter_left = Counter(left_side.dropna()) # dic
    counter_right = Counter(right_side.dropna())
    
    # 计算两侧相同元素的个数，确保每个元素只计算一次
    matches = sum((counter_left & counter_right).values())
    total = sum(counter_left.values())
    
    # char_level
    counter_left_char = Counter(''.join(left_side.dropna().tolist()))
    counter_right_char = Counter(''.join(right_side.dropna().tolist()))
    matches_char = sum((counter_left_char & counter_right_char).values())
    total_char = sum(counter_left_char.values())
    return [matches, total - matches, total_char-matches_char]

In [46]:
xlsx_file_path = '../../data/human/MNREADChartScoreSheet.xlsx'
WORD_NUM = 15
df_human = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM,human=True,L_R=True)
df_human[['match','missing','missing_char']] = df_human.apply(count_matches_per_row, axis=1, result_type='expand')
df_human.reset_index(inplace=True)
df_human['missing_clipped'] = df_human['missing'].clip(upper=10)
df_human = df_human.groupby(['image_no','a','b','SubID']).apply(lambda x: -0.2 + x['missing_clipped'].sum()*0.01).reset_index()
df_human.rename(columns={0:'acuity'}, inplace=True)
df_human['a'] = df_human['a'].round(3)
df_human['b'] = df_human['b'].round(3)
df_human['a'].replace(0.156, 0.157, inplace=True)
df_human['b'].replace(0.156, 0.157, inplace=True)
df_human['a'].replace(0.287, 0.288, inplace=True)
df_human['b'].replace(0.287, 0.288, inplace=True)
df_human = df_human.groupby(['image_no','a','b']).apply(lambda x: x['acuity'].mean())
df_human = df_human.reset_index()
df_human.rename(columns={0:'acuity'}, inplace=True)
df_human = df_human.groupby(['a','b']).apply(lambda x: x['acuity'].mean())
df_human = df_human.reset_index().rename(columns={0:'acuity'})
df_human.to_csv('../../data/human/human_mnread_acuity.csv',index=False)

/tmp/ipykernel_3157193/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/tmp/ipykernel_3157193/3200305329.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_human = df_human.groupby(['image_no','a','b','SubID']).apply(lambda

In [45]:
xlsx_file_path = '../../data/human/LetterChartScoreSheet.xlsx'
WORD_NUM = 5
df_human = get_gt(file_path=xlsx_file_path, word_num=WORD_NUM,human=True,L_R=True)
df_human[['match','missing','missing_char']] = df_human.apply(count_matches_per_row, axis=1, result_type='expand')
df_human.reset_index(inplace=True)
df_human['missing_clipped'] = df_human['missing'].clip(upper=10)
df_human = df_human.groupby(['image_no','a','b','SubID']).apply(lambda x: -0.2 + x['missing_clipped'].sum()*0.01).reset_index()
df_human.rename(columns={0:'acuity'}, inplace=True)
df_human['a'] = df_human['a'].round(3)
df_human['b'] = df_human['b'].round(3)
df_human['a'].replace(0.156, 0.157, inplace=True)
df_human['b'].replace(0.156, 0.157, inplace=True)
df_human['a'].replace(0.287, 0.288, inplace=True)
df_human['b'].replace(0.287, 0.288, inplace=True)
df_human = df_human.groupby(['image_no','a','b']).apply(lambda x: x['acuity'].mean())
df_human = df_human.reset_index()
df_human.rename(columns={0:'acuity'}, inplace=True)
df_human = df_human.groupby(['a','b']).apply(lambda x: x['acuity'].mean())
df_human = df_human.reset_index().rename(columns={0:'acuity'})
df_human.to_csv('../../data/human/human_etdrs_acuity.csv',index=False)

/tmp/ipykernel_3157193/999310692.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gt['row'] = gt.groupby(['image_no','SubID']).apply(lambda x: np.round((x['print_size'].max()-x['print_size']+0.1)*10)).reset_index().set_index('level_2')['print_size']
/tmp/ipykernel_3157193/466544350.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_human = df_human.groupby(['image_no','a','b','SubID']).apply(lambda 